In [2]:
### LLM 모델 기본 사용법과 프롬프트 전달방법 01
### Ollama 로컬 모델 사용

from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma4:e4b")

# llm.invoke("안녕하세요")
# llm.invoke("대한민국의 수도는?") # -> 이게 바로 프롬프트, 모델에게 전달하는 명령어 이자 질의 그자체


In [4]:
### 프롬프트 템플릿 사용

from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template="""
    {country}의 수도는?
    """,
    input_variables=["country"] ,
)

prompt_template.invoke({"country": "대한민국"}) # StringPromptValue(text='\n    대한민국의 수도는?\n    ')

llm.invoke(prompt_template.invoke({"country": "대한민국"}))



AIMessage(content='대한민국의 수도는 **서울**입니다.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-07-19T23:27:12.659614Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11945717000, 'load_duration': 7035650125, 'prompt_eval_count': 22, 'prompt_eval_duration': 164114000, 'eval_count': 166, 'eval_duration': 4716607000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019f7cb4-5467-74d2-ba2a-337a72b035a5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 166, 'total_tokens': 188})

In [ ]:
### langchain의 다앙햔 메세지 형식

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

message_list = [
    SystemMessage(content="당신은 나라별 수도 전문가입니다."), #시스템의 정체성 페르소나
    HumanMessage(content="대한민국의 수도는?"), # 사용자의 질의
    AIMessage(content="서울입니다."), # 모델의 답변
    HumanMessage(content="미국의 수도는?"), # 사용자의 질의
    AIMessage(content="워싱턴 D.C.입니다.") # 모델의 답변
] ## 마치 대화 이력이 있던 것처럼 HumanMessage, AIMessage 를 차례로 여러건 넣는방법
## 이걸 few-shot 프롬프트 라고함.
## zero-shot 보다 few-shot 이 통계적으로 더 정확한 답변 혹은 질의 의도에 맞는 답변을 한다고함.

llm.invoke(message_list)



AIMessage(content='워싱턴 D.C.입니다.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-07-19T22:38:44.346942Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4739343250, 'load_duration': 331669750, 'prompt_eval_count': 63, 'prompt_eval_duration': 703449000, 'eval_count': 126, 'eval_duration': 3656593000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019f7c88-0fde-7a10-8329-e2ddbc0e7a61-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 126, 'total_tokens': 189})

In [ ]:
### 문자열 응답 정제 StrOutputParser

from langchain_core.output_parsers import StrOutputParser

prompt_template = PromptTemplate(
    template="""
    {country}의 수도는? 수도 이름만 답해줘
    """,
    input_variables=["country"]
)

prompt = prompt_template.invoke({"country": "일본"})
print(prompt)

llm.invoke(prompt_template.invoke({"country": "프랑스"}))
output_parser = StrOutputParser()

answer = output_parser.invoke(llm.invoke(prompt_template.invoke({"country": "프랑스"})))
print(answer)



text='\n    일본의 수도는? 수도 이름만 답해줘\n    '
파리


In [ ]:
### JSON 응답 정제 JsonOutputParser -> 비추 잘안됨.

from langchain_core.output_parsers import JsonOutputParser

prompt_template = PromptTemplate(
    template="""
    {country}의 수도는? 수도 이름만 답해줘
    """,
    input_variables=["country"]
)

output_parser = JsonOutputParser()

answer = output_parser.invoke(llm.invoke(prompt_template.invoke({"country":"배트남"})))
print(answer)

### 에러가 나는데, 타입 에러가 남. 그런데 애초에 JsonOutputParser 로 invoke 해서 쓴건데 타입 에러가 난다는것 부터 쓰기 어려운 매소드임.
### 실제 결과가 JSON 모양의 String으로 와서 생기는 문제


OutputParserException: Invalid json output: 하노이
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [9]:
### 따라서 구조체 형식으로 뽑고 싶다면
### JsonOutputParser 가 아닌 llm.with_structured_output과 Pydantic Class를 같이 사용하면됨.

from typing import cast
from pydantic import BaseModel, Field

class CountryDetail(BaseModel):
    capital: str = Field(description="한 국가의 수도")

structured_llm = llm.with_structured_output(CountryDetail)

prompt_template = PromptTemplate(
    template="""
    {country}의 수도는? 수도 이름만 답해줘
    JSON 형식으로 답해줘
    """,
    input_variables=["country"]
)

llm.invoke(prompt_template.invoke({"country":"배트남"}))
resData:CountryDetail = cast(CountryDetail, structured_llm.invoke(prompt_template.invoke({"country":"배트남"})))
print(resData.capital)
print(resData.model_dump)




하노이
<bound method BaseModel.model_dump of CountryDetail(capital='하노이')>
